In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load dataset
DATA_PATH = "../data/raw/DatasetIcotMond.xlsx"
df = pd.read_excel(DATA_PATH)

# Clean column names
df.columns = df.columns.str.strip()

# =========================
# Biomechanical groups
# =========================
RHYTHMIC_STABILITY = [
    "HR V", "HR ML", "HR AP",
    "iHR V", "iHR ML", "iHR AP",
    "%det V", "%det ML", "%det AP"
]

COMPLEXITY = [
    "MSE V", "MSE ML", "MSE AP"
]

POSTURAL_ORIENTATION = [
    "Tilt", "Obliquity", "Rotation (range)"
]

ALL_EXPOSURES = RHYTHMIC_STABILITY + COMPLEXITY + POSTURAL_ORIENTATION

# Sanity check
missing = [c for c in ALL_EXPOSURES if c not in df.columns]
assert len(missing) == 0, f"Missing biomechanical variables: {missing}"

print("Dataset shape:", df.shape)
print("Biomechanical variables:", len(ALL_EXPOSURES))

In [ ]:
# Subset biomechanical data
X_raw = df[ALL_EXPOSURES].copy()

# Convert to numeric
X_raw = X_raw.apply(pd.to_numeric, errors="coerce")

# Report missingness
missing_rate = X_raw.isna().mean().sort_values(ascending=False)
print("Missing rate (%):")
display((missing_rate * 100).round(2))

# Impute (median, conservative)
imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(
    imputer.fit_transform(X_raw),
    columns=X_raw.columns,
    index=X_raw.index
)

# Standardize
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X_imputed),
    columns=X_imputed.columns,
    index=X_imputed.index
)

print("Final exposure matrix shape:", X_scaled.shape)

In [ ]:
plt.figure(figsize=(12,10))
sns.heatmap(
    X_scaled.corr(),
    cmap="coolwarm",
    center=0,
    square=True
)
plt.title("Correlation matrix of trunk biomechanical exposures")
plt.show()

In [ ]:
pca_rhythm = PCA(n_components=3, random_state=42)
X_rhythm = X_scaled[RHYTHMIC_STABILITY]

pca_rhythm.fit(X_rhythm)

# Explained variance
plt.plot(np.cumsum(pca_rhythm.explained_variance_ratio_), marker="o")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA – Rhythmic Stability")
plt.show()

# Loadings
loadings_rhythm = pd.DataFrame(
    pca_rhythm.components_.T,
    index=RHYTHMIC_STABILITY,
    columns=[f"PC{i+1}" for i in range(pca_rhythm.n_components_)]
)

display(loadings_rhythm)

# Keep PC1
df["PC1_Rhythmic"] = pca_rhythm.transform(X_rhythm)[:, 0]

In [ ]:
pca_complexity = PCA(n_components=2, random_state=42)
X_complexity = X_scaled[COMPLEXITY]

pca_complexity.fit(X_complexity)

plt.plot(np.cumsum(pca_complexity.explained_variance_ratio_), marker="o")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA – Neuromotor Complexity")
plt.show()

loadings_complexity = pd.DataFrame(
    pca_complexity.components_.T,
    index=COMPLEXITY,
    columns=[f"PC{i+1}" for i in range(pca_complexity.n_components_)]
)

display(loadings_complexity)

df["PC1_Complexity"] = pca_complexity.transform(X_complexity)[:, 0]

In [ ]:
pca_posture = PCA(n_components=2, random_state=42)
X_posture = X_scaled[POSTURAL_ORIENTATION]

pca_posture.fit(X_posture)

plt.plot(np.cumsum(pca_posture.explained_variance_ratio_), marker="o")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA – Postural Orientation")
plt.show()

loadings_posture = pd.DataFrame(
    pca_posture.components_.T,
    index=POSTURAL_ORIENTATION,
    columns=[f"PC{i+1}" for i in range(pca_posture.n_components_)]
)

display(loadings_posture)

df["PC1_Postural"] = pca_posture.transform(X_posture)[:, 0]

In [ ]:
latent_axes = ["PC1_Rhythmic", "PC1_Complexity", "PC1_Postural"]

plt.figure(figsize=(5,4))
sns.heatmap(
    df[latent_axes].corr(),
    annot=True,
    cmap="coolwarm",
    center=0
)
plt.title("Correlation between latent biomechanical axes")
plt.show()

In [ ]:
# Age strata
df["Age_group"] = pd.qcut(df["Age"], q=3)

fig, axes = plt.subplots(1, 3, figsize=(15,4))
for ax, pc in zip(axes, latent_axes):
    sns.boxplot(x="Age_group", y=pc, data=df, ax=ax)
    ax.set_title(f"{pc} across age strata")
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
OUTPUT_COLS = latent_axes

df_latent = df[OUTPUT_COLS].copy()
df_latent.to_csv("../data/processed/latent_trunk_axes.csv", index=False)

print("Saved latent biomechanical axes:")
display(df_latent.describe())

## Bootstrap PCA + CI delle loadings

In [ ]:
# ============================================================
# CELL 1 — Bootstrap PCA loadings + 95% CI
# Latent biomechanical axes stability
# ============================================================

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# ----------------------------
# PARAMETERS
# ----------------------------
N_BOOT = 500
RANDOM_STATE = 42

# ----------------------------
# VARIABLE GROUPS
# ----------------------------
rhythmic_vars = [
    "HR V", "HR ML", "HR AP",
    "iHR V", "iHR ML", "iHR AP",
    "%det V", "%det ML", "%det AP"
]

complexity_vars = ["MSE V", "MSE ML", "MSE AP"]

postural_vars = ["Tilt", "Obliquity", "Rotation (range)"]

# ----------------------------
# BOOTSTRAP PCA FUNCTION
# ----------------------------
def bootstrap_pca_loadings(
    df,
    variables,
    n_components=1,
    n_boot=500,
    random_state=42
):
    """
    Bootstrap PCA loadings with 95% CI.
    Complete-case analysis per variable group.
    """

    # Complete-case subset
    clean_df = df[variables].dropna()

    if clean_df.shape[0] < 50:
        raise ValueError(
            f"Too few complete cases for PCA bootstrap: {clean_df.shape[0]}"
        )

    loadings = []

    for b in range(n_boot):
        sample = clean_df.sample(
            frac=1,
            replace=True,
            random_state=random_state + b
        )

        X = StandardScaler().fit_transform(sample.values)

        pca = PCA(n_components=n_components)
        pca.fit(X)

        loadings.append(pca.components_[0])

    loadings = np.array(loadings)

    ci_df = pd.DataFrame({
        "mean_loading": loadings.mean(axis=0),
        "ci_lower": np.percentile(loadings, 2.5, axis=0),
        "ci_upper": np.percentile(loadings, 97.5, axis=0)
    }, index=variables)

    return ci_df


# ----------------------------
# RUN BOOTSTRAP PCA
# ----------------------------
ci_rhythmic = bootstrap_pca_loadings(
    df, rhythmic_vars, n_boot=N_BOOT
)

ci_complexity = bootstrap_pca_loadings(
    df, complexity_vars, n_boot=N_BOOT
)

ci_postural = bootstrap_pca_loadings(
    df, postural_vars, n_boot=N_BOOT
)

# ----------------------------
# DISPLAY RESULTS
# ----------------------------
print("=== Rhythmic Stability Axis ===")
display(ci_rhythmic)

print("\n=== Neuromotor Complexity Axis ===")
display(ci_complexity)

print("\n=== Postural Orientation Axis ===")
display(ci_postural)

In [ ]:
# =========================================
# Variance explained by PC1 (Table 2 output)
# =========================================

variance_table = pd.DataFrame({
    "Domain": ["Rhythmic stability", "Neuromotor complexity", "Postural orientation"],
    "N_features": [
        len(RHYTHMIC_STABILITY),
        len(COMPLEXITY),
        len(POSTURAL_ORIENTATION)
    ],
    "PC1_variance_explained_%": [
        pca_rhythm.explained_variance_ratio_[0] * 100,
        pca_complexity.explained_variance_ratio_[0] * 100,
        pca_posture.explained_variance_ratio_[0] * 100
    ]
}).round(1)

variance_table

In [ ]:
# =========================================
# Correlation between latent axes
# =========================================

latent_corr = df[latent_axes].corr().round(2)
latent_corr

In [ ]:
def summarize_ci(ci_df):
    return pd.DataFrame({
        "CI_lower_min": [ci_df["ci_lower"].min()],
        "CI_upper_max": [ci_df["ci_upper"].max()]
    })

stability_summary = pd.concat([
    summarize_ci(ci_rhythmic).assign(Domain="Rhythmic stability"),
    summarize_ci(ci_complexity).assign(Domain="Neuromotor complexity"),
    summarize_ci(ci_postural).assign(Domain="Postural orientation")
])

stability_summary

In [ ]:
# ============================================================
# FINAL FIGURE — PANEL C
# Latent biomechanical axis construction (PCA)
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl

# ----------------------------
# GLOBAL STYLE — Times New Roman
# ----------------------------
mpl.rcParams["font.family"] = "Times New Roman"
mpl.rcParams["axes.titlesize"] = 12
mpl.rcParams["axes.labelsize"] = 10
mpl.rcParams["xtick.labelsize"] = 9
mpl.rcParams["ytick.labelsize"] = 9

# ----------------------------
# INPUT: bootstrap CI tables
# ----------------------------
PANELS = [
    ("Rhythmic stability", ci_rhythmic),
    ("Neuromotor complexity", ci_complexity),
    ("Postural orientation", ci_postural),
]

# ----------------------------
# FIGURE SETUP
# ----------------------------
fig, axes = plt.subplots(
    1, 3,
    figsize=(15, 4),
    sharey=False
)

for ax, (title, ci_df) in zip(axes, PANELS):

    # ------------------------------------------------
    # Sort variables by absolute PC1 loading (descending)
    # ------------------------------------------------
    ci_df_sorted = ci_df.reindex(
        ci_df["mean_loading"].abs().sort_values(ascending=True).index
    )

    y_pos = np.arange(len(ci_df_sorted))
    mean = ci_df_sorted["mean_loading"].values
    lower = ci_df_sorted["ci_lower"].values
    upper = ci_df_sorted["ci_upper"].values

    # ----------------------------
    # Horizontal bars (PC1 loadings)
    # ----------------------------
    ax.barh(
        y_pos,
        mean,
        color="#4C72B0",
        alpha=0.85
    )

    # ----------------------------
    # Confidence intervals
    # ----------------------------
    ax.hlines(
        y=y_pos,
        xmin=lower,
        xmax=upper,
        color="black",
        linewidth=1
    )

    ax.axvline(0, color="black", linewidth=0.8)

    # ----------------------------
    # Axis labels and ticks
    # ----------------------------
    ax.set_yticks(y_pos)
    ax.set_yticklabels(ci_df_sorted.index)
    ax.set_xlabel("PC1 loading")

    ax.set_title(title, fontweight="bold")

    # ----------------------------
    # Clean spines (npj-style)
    # ----------------------------
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# ----------------------------
# GLOBAL TITLES
# ----------------------------
fig.suptitle(
    "Latent biomechanical axis construction (PCA)",
    fontsize=14,
    fontweight="bold"
)

fig.text(
    0.5, -0.08,
    "Domain-specific dimensionality reduction: first principal component retained per domain",
    ha="center",
    fontsize=10
)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# Data (posterior summaries)
# -----------------------------
labels = ["PC1_Rhythmic", "PC1_Complexity"]

means = np.array([0.154, 0.051])
hdi_low = np.array([0.044, -0.069])
hdi_high = np.array([0.272, 0.168])

y_pos = np.arange(len(labels))

# -----------------------------
# Global style (npj / Nature)
# -----------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 11,
    "axes.linewidth": 0.8,
})

# -----------------------------
# Figure
# -----------------------------
fig, ax = plt.subplots(figsize=(6.2, 2.4))

# Horizontal CI bars
ax.hlines(
    y=y_pos,
    xmin=hdi_low,
    xmax=hdi_high,
    color="black",
    linewidth=2,
)

# Posterior means
ax.plot(
    means,
    y_pos,
    "o",
    color="black",
    markersize=6,
)

# Reference line at zero
ax.axvline(
    x=0,
    color="gray",
    linestyle="--",
    linewidth=1,
    alpha=0.6,
)

# -----------------------------
# Axes formatting
# -----------------------------
ax.set_yticks(y_pos)
ax.set_yticklabels(labels)
ax.invert_yaxis()  # top = most relevant

ax.set_xlabel("Posterior loading (mean ± 95% HDI)")
ax.set_title("Posterior loadings of trunk biomechanical axes on latent motor severity")

# Clean spines
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Tight layout
plt.tight_layout()

# -----------------------------
# Save (journal-ready)
# -----------------------------
plt.savefig("Fig_X_forest_loadings.pdf", dpi=300, bbox_inches="tight")
plt.savefig("Fig_X_forest_loadings.svg", bbox_inches="tight")

plt.show()

In [ ]:
# ============================================================
# ANCHOR JUSTIFICATION ANALYSIS
# Statistical criteria for selecting anchor domain
# ============================================================

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

def domain_diagnostics(df, variables, n_boot=300):
    """
    Compute objective statistical criteria:
    - PC1 explained variance
    - Eigenvalue dominance ratio (λ1 / λ2)
    - Signal-to-noise ratio
    - Bootstrap loading stability (mean SD)
    """

    clean_df = df[variables].dropna()
    X = StandardScaler().fit_transform(clean_df)

    pca = PCA()
    pca.fit(X)

    eigenvalues = pca.explained_variance_
    var_explained = pca.explained_variance_ratio_[0]

    # Eigenvalue dominance ratio
    if len(eigenvalues) > 1:
        dominance_ratio = eigenvalues[0] / eigenvalues[1]
    else:
        dominance_ratio = np.nan

    # Signal-to-noise ratio (PC1 variance vs rest)
    snr = eigenvalues[0] / np.sum(eigenvalues[1:])

    # Bootstrap loading stability
    boot_loadings = []

    for b in range(n_boot):
        sample = clean_df.sample(frac=1, replace=True, random_state=42 + b)
        Xb = StandardScaler().fit_transform(sample)
        pca_b = PCA(n_components=1)
        pca_b.fit(Xb)
        boot_loadings.append(pca_b.components_[0])

    boot_loadings = np.array(boot_loadings)
    loading_sd = boot_loadings.std(axis=0).mean()  # mean instability

    return {
        "PC1_variance_explained": var_explained,
        "Eigenvalue_dominance_ratio": dominance_ratio,
        "Signal_to_noise_ratio": snr,
        "Mean_bootstrap_loading_SD": loading_sd,
        "N_features": len(variables),
        "N_complete_cases": clean_df.shape[0]
    }


# Run diagnostics for each domain
diag_rhythm = domain_diagnostics(df, RHYTHMIC_STABILITY)
diag_complexity = domain_diagnostics(df, COMPLEXITY)
diag_postural = domain_diagnostics(df, POSTURAL_ORIENTATION)

anchor_table = pd.DataFrame([
    diag_rhythm,
    diag_complexity,
    diag_postural
], index=[
    "Rhythmic stability",
    "Neuromotor complexity",
    "Lower trunk kinematics"
])

anchor_table.round(3)